# Checkpoint-aware representation comparison

Use the next cell for the current analysis. It compares identical ordered images across pretrained, trained-head, raw, and grid-cropped representations. The older prototype cells below are retained for reference, but they overwrite the DINO processor with the SAM processor and should not be run as part of this comparison.

In [ ]:
from analysis.clustering.compare import run

results = run("analysis/clustering/config.toml")
results

In [30]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import math
import itertools
import torch
from pathlib import Path
import torch.nn.functional as F

In [31]:
PATH='/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set'
FILE = 'RSFB-Phenotyping_training_set_scores.csv'

In [32]:
data = pd.read_csv(os.path.join(PATH, FILE))
data.head()

,Filename,Score_JLU,Score_GAU,mean_score
0,20251021_120858,10,10.5,10.25
1,20251021_120905,13,16.5,14.75
2,20251021_120912,9,6.5,7.75
3,20251021_120926,23,19.5,21.25
4,20251021_120933,20,21.5,20.75


In [33]:
images_name = list(data['Filename'])
label = list(data['mean_score'])

In [34]:
image_paths = list(Path(PATH).glob("**/*"))
image_paths = [p for p in image_paths if p.suffix.lower() == ".jpg"]

In [35]:
image_paths

[PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_132849.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_160640.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_160221.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_162906.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_162202.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSFB-Phenotyping_training_set/20251021_155541.jpg'),
 PosixPath('/home/nfs/data/nvme_datasets/Pictures_CFSB_leaf_damage/RSFB-Phenotyping_training_set/RSF

In [36]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image



processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m")
dino_model = AutoModel.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m", device_map="auto")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 6566.53it/s]


In [37]:
# Detection is performed on a resized copy for speed.
MAX_DETECTION_DIMENSION = 1600

# Final width and height of each extracted square.
CELL_SIZE = 900

# Removes the steel bars from the borders of each extracted cell.
INNER_MARGIN_FRACTION = 0.05

# Steel-color thresholds.
# Steel is generally less saturated than brown soil.
SATURATION_MAX = 60
VALUE_MIN = 70

# Maximum allowed deviation from horizontal or vertical.
ANGLE_TOLERANCE = 14.0

In [38]:
def resize_for_detection(
    image: np.ndarray,
    maximum_dimension: int,
) -> tuple[np.ndarray, float]:
    height, width = image.shape[:2]

    scale = min(1.0, maximum_dimension / max(height, width))

    if scale == 1.0:
        return image.copy(), scale

    resized = cv2.resize(
        image,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_AREA,
    )

    return resized, scale


def detect_possible_steel(
    image: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Build a permissive steel mask and an edge mask for Hough fallback.

    The primary detector below uses the source image directly. This mask is
    deliberately permissive because polished steel changes brightness and hue
    along a bar as it reflects the sky and soil.
    """
    minimum_dimension = min(image.shape[:2])

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]

    steel_color_mask = (
        (saturation < SATURATION_MAX)
        & (value > VALUE_MIN)
    ).astype(np.uint8) * 255

    close_size = max(3, int(round(minimum_dimension * 0.004)))
    if close_size % 2 == 0:
        close_size += 1

    steel_color_mask = cv2.morphologyEx(
        steel_color_mask,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_size, close_size),
        ),
    )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8),
    ).apply(gray)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    median_intensity = float(np.median(gray))
    lower = int(max(20, 0.55 * median_intensity))
    upper = int(min(220, max(lower + 30, 1.45 * median_intensity)))

    edges = cv2.Canny(gray, lower, upper)

    support_size = max(3, int(round(minimum_dimension * 0.007)))
    if support_size % 2 == 0:
        support_size += 1

    color_support = cv2.dilate(
        steel_color_mask,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (support_size, support_size),
        ),
    )

    steel_line_mask = cv2.bitwise_and(edges, color_support)
    return steel_color_mask, steel_line_mask


def _segments_from_lsd(
    source_image: np.ndarray,
) -> list[tuple[float, float, tuple[int, int, int, int]]]:
    """Detect long, coherent segments without assuming exact axis alignment."""
    if source_image.ndim == 3:
        gray = cv2.cvtColor(source_image, cv2.COLOR_BGR2GRAY)
    else:
        gray = source_image.copy()

    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    minimum_dimension = min(gray.shape[:2])

    detector = cv2.createLineSegmentDetector(cv2.LSD_REFINE_ADV)
    detected = detector.detect(gray)[0]

    if detected is None:
        return []

    minimum_length = max(35.0, minimum_dimension * 0.04)
    segments = []

    for x1, y1, x2, y2 in np.asarray(detected).reshape(-1, 4):
        dx = float(x2 - x1)
        dy = float(y2 - y1)
        length = math.hypot(dx, dy)

        if length < minimum_length:
            continue

        angle = math.degrees(math.atan2(dy, dx)) % 180.0

        segments.append(
            (
                length,
                angle,
                (
                    int(round(x1)),
                    int(round(y1)),
                    int(round(x2)),
                    int(round(y2)),
                ),
            )
        )

    return segments


def _segments_from_hough(
    line_mask: np.ndarray,
) -> list[tuple[float, float, tuple[int, int, int, int]]]:
    """Compatibility fallback for images where LSD returns too few segments."""
    height, width = line_mask.shape[:2]
    minimum_dimension = min(height, width)

    detected = cv2.HoughLinesP(
        line_mask,
        rho=1,
        theta=np.pi / 720,
        threshold=max(25, int(round(minimum_dimension * 0.03))),
        minLineLength=max(40, int(round(minimum_dimension * 0.05))),
        maxLineGap=max(20, int(round(minimum_dimension * 0.04))),
    )

    if detected is None:
        return []

    segments = []

    for x1, y1, x2, y2 in np.asarray(detected).reshape(-1, 4):
        dx = float(x2 - x1)
        dy = float(y2 - y1)
        length = math.hypot(dx, dy)
        angle = math.degrees(math.atan2(dy, dx)) % 180.0

        segments.append(
            (
                length,
                angle,
                (int(x1), int(y1), int(x2), int(y2)),
            )
        )

    return segments


def detect_line_segments(
    line_mask: np.ndarray,
    source_image: np.ndarray | None = None,
) -> list[tuple[float, float, tuple[int, int, int, int]]]:
    """Return segments in the same format as the original implementation.

    Passing ``source_image`` enables the stronger LSD path. Calling this with
    only ``line_mask`` remains supported and uses the previous Hough-style path.
    """
    if source_image is not None:
        segments = _segments_from_lsd(source_image)
        if segments:
            return segments

    segments = _segments_from_hough(line_mask)

    if not segments:
        raise RuntimeError(
            "No sufficiently long steel-line candidates were detected."
        )

    return segments


def _signed_orientation_error(angle: float, orientation: str) -> float:
    if orientation == "horizontal":
        return angle if angle <= 90.0 else angle - 180.0
    if orientation == "vertical":
        return angle - 90.0
    raise ValueError("orientation must be 'vertical' or 'horizontal'")


def cluster_similar_lines(
    segments: list[tuple[float, float, tuple[int, int, int, int]]],
    image_shape: tuple[int, int],
    orientation: str,
) -> list[dict]:
    """Group fragments belonging to the same projected steel bar."""
    height, width = image_shape
    minimum_dimension = min(height, width)
    center_x = width / 2.0
    center_y = height / 2.0

    candidates = []

    for length, angle, segment in segments:
        orientation_error = _signed_orientation_error(angle, orientation)

        if abs(orientation_error) > ANGLE_TOLERANCE:
            continue

        x1, y1, x2, y2 = segment

        if orientation == "vertical":
            if abs(y2 - y1) < 1:
                continue
            position = x1 + (center_y - y1) * (x2 - x1) / (y2 - y1)
        else:
            if abs(x2 - x1) < 1:
                continue
            position = y1 + (center_x - x1) * (y2 - y1) / (x2 - x1)

        # Lines far outside the image are extrapolation artifacts.
        relevant_dimension = width if orientation == "vertical" else height
        if not -0.15 * relevant_dimension <= position <= 1.15 * relevant_dimension:
            continue

        candidates.append(
            (
                float(position),
                float(length),
                segment,
                float(orientation_error),
            )
        )

    candidates.sort(key=lambda item: item[0])

    position_tolerance = minimum_dimension * 0.018
    angle_tolerance = max(4.0, ANGLE_TOLERANCE * 0.6)
    groups: list[list] = []

    for candidate in candidates:
        best_group = None
        best_cost = float("inf")

        for group_index, group in enumerate(groups):
            weights = [item[1] for item in group]
            group_position = float(np.average(
                [item[0] for item in group],
                weights=weights,
            ))
            group_angle = float(np.average(
                [item[3] for item in group],
                weights=weights,
            ))

            position_distance = abs(candidate[0] - group_position)
            angle_distance = abs(candidate[3] - group_angle)

            if (
                position_distance <= position_tolerance
                and angle_distance <= angle_tolerance
            ):
                cost = (
                    position_distance / position_tolerance
                    + angle_distance / angle_tolerance
                )
                if cost < best_cost:
                    best_group = group_index
                    best_cost = cost

        if best_group is None:
            groups.append([candidate])
        else:
            groups[best_group].append(candidate)

    clusters = []

    for group in groups:
        weights = [item[1] for item in group]
        clusters.append(
            {
                "position": float(np.average(
                    [item[0] for item in group],
                    weights=weights,
                )),
                "support": float(sum(weights)),
                "angle": float(np.average(
                    [item[3] for item in group],
                    weights=weights,
                )),
                "segments": [item[2] for item in group],
            }
        )

    return sorted(clusters, key=lambda cluster: cluster["position"])


def choose_three_grid_lines(
    clusters: list[dict],
    dimension: int,
) -> list[dict]:
    """Select three strong, approximately equally spaced projected bars."""
    strongest = sorted(
        clusters,
        key=lambda cluster: cluster["support"],
        reverse=True,
    )[:12]
    strongest.sort(key=lambda cluster: cluster["position"])

    best_result = None

    for candidate_triplet in itertools.combinations(strongest, 3):
        positions = [cluster["position"] for cluster in candidate_triplet]
        total_span = positions[2] - positions[0]

        if not dimension * 0.18 <= total_span <= dimension * 0.82:
            continue

        first_spacing = positions[1] - positions[0]
        second_spacing = positions[2] - positions[1]

        if first_spacing <= 0 or second_spacing <= 0:
            continue

        spacing_error = abs(first_spacing - second_spacing) / total_span

        if spacing_error > 0.32:
            continue

        total_support = sum(cluster["support"] for cluster in candidate_triplet)
        mean_angle = float(np.average(
            [cluster.get("angle", 0.0) for cluster in candidate_triplet],
            weights=[cluster["support"] for cluster in candidate_triplet],
        ))
        angle_spread = max(
            abs(cluster.get("angle", 0.0) - mean_angle)
            for cluster in candidate_triplet
        )

        # Equal spacing matters strongly; angle consistency helps reject label
        # edges and long soil/straw features.
        score = total_support * math.exp(
            -4.0 * spacing_error
            -0.08 * angle_spread
        )

        if best_result is None or score > best_result[0]:
            best_result = (score, candidate_triplet)

    if best_result is None:
        cluster_information = [
            (
                round(cluster["position"], 1),
                round(cluster["support"], 1),
                round(cluster.get("angle", 0.0), 1),
            )
            for cluster in strongest
        ]
        raise RuntimeError(
            "Could not find three equally spaced steel bars. "
            "Detected clusters (position, support, angle): "
            f"{cluster_information}"
        )

    return list(best_result[1])


def fit_infinite_line(cluster: dict) -> np.ndarray:
    """Fit a robust homogeneous line a*x + b*y + c = 0."""
    points = []

    for x1, y1, x2, y2 in cluster["segments"]:
        points.append((x1, y1))
        points.append((x2, y2))

    points_array = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)

    vx, vy, x0, y0 = cv2.fitLine(
        points_array,
        cv2.DIST_HUBER,
        0,
        0.01,
        0.01,
    ).reshape(-1)

    a = float(vy)
    b = float(-vx)
    c = -(a * float(x0) + b * float(y0))
    normalization = math.hypot(a, b)

    return np.array(
        [a / normalization, b / normalization, c / normalization],
        dtype=np.float64,
    )


def line_intersection(
    first_line: np.ndarray,
    second_line: np.ndarray,
) -> np.ndarray:
    intersection = np.cross(first_line, second_line)

    if abs(intersection[2]) < 1e-8:
        raise RuntimeError("Detected steel lines are parallel or invalid.")

    return np.array(
        [
            intersection[0] / intersection[2],
            intersection[1] / intersection[2],
        ],
        dtype=np.float32,
    )


def _select_grid_lines(
    segments: list[tuple[float, float, tuple[int, int, int, int]]],
    image_shape: tuple[int, int],
) -> tuple[list[dict], list[dict]]:
    vertical_clusters = cluster_similar_lines(
        segments,
        image_shape,
        orientation="vertical",
    )
    horizontal_clusters = cluster_similar_lines(
        segments,
        image_shape,
        orientation="horizontal",
    )

    selected_vertical = choose_three_grid_lines(
        vertical_clusters,
        image_shape[1],
    )
    selected_horizontal = choose_three_grid_lines(
        horizontal_clusters,
        image_shape[0],
    )

    return selected_vertical, selected_horizontal


def _render_grid_mask(
    image_shape: tuple[int, int],
    grid_points: np.ndarray,
) -> np.ndarray:
    mask = np.zeros(image_shape, dtype=np.uint8)
    thickness = max(2, int(round(min(image_shape) * 0.005)))

    for row in range(3):
        start = tuple(np.round(grid_points[row, 0]).astype(int))
        end = tuple(np.round(grid_points[row, 2]).astype(int))
        cv2.line(mask, start, end, 255, thickness)

    for column in range(3):
        start = tuple(np.round(grid_points[0, column]).astype(int))
        end = tuple(np.round(grid_points[2, column]).astype(int))
        cv2.line(mask, start, end, 255, thickness)

    return mask


def detect_grid(
    image: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Detect the 3x3 intersections while preserving the original API."""
    if image is None or image.ndim != 3:
        raise ValueError("detect_grid expects a valid BGR color image")

    detection_image, scale = resize_for_detection(
        image,
        MAX_DETECTION_DIMENSION,
    )

    _, candidate_mask = detect_possible_steel(detection_image)

    # Primary path: perspective- and occlusion-tolerant LSD segments.
    segments = detect_line_segments(
        candidate_mask,
        source_image=detection_image,
    )

    try:
        selected_vertical, selected_horizontal = _select_grid_lines(
            segments,
            candidate_mask.shape,
        )
    except RuntimeError as primary_error:
        # Fallback: supplement LSD with Hough candidates from the color/edge mask.
        hough_segments = _segments_from_hough(candidate_mask)
        try:
            selected_vertical, selected_horizontal = _select_grid_lines(
                segments + hough_segments,
                candidate_mask.shape,
            )
        except RuntimeError as fallback_error:
            raise RuntimeError(
                "Grid detection failed with both LSD and Hough candidates. "
                f"LSD result: {primary_error}; fallback result: {fallback_error}"
            ) from fallback_error

    vertical_lines = [
        fit_infinite_line(cluster)
        for cluster in selected_vertical
    ]
    horizontal_lines = [
        fit_infinite_line(cluster)
        for cluster in selected_horizontal
    ]

    detection_grid_points = np.zeros((3, 3, 2), dtype=np.float32)

    for row, horizontal_line in enumerate(horizontal_lines):
        for column, vertical_line in enumerate(vertical_lines):
            detection_grid_points[row, column] = line_intersection(
                horizontal_line,
                vertical_line,
            )

    if not np.isfinite(detection_grid_points).all():
        raise RuntimeError("Grid intersections contain invalid coordinates.")

    outer_corners = np.float32([
        detection_grid_points[0, 0],
        detection_grid_points[0, 2],
        detection_grid_points[2, 2],
        detection_grid_points[2, 0],
    ])
    grid_area = abs(cv2.contourArea(outer_corners))

    if grid_area < 0.025 * candidate_mask.size:
        raise RuntimeError(
            "Detected lines form an implausibly small grid; refusing the result."
        )

    steel_line_mask = _render_grid_mask(
        candidate_mask.shape,
        detection_grid_points,
    )
    grid_points = detection_grid_points / scale

    return grid_points, detection_image, steel_line_mask



def warp_cell(
    image: np.ndarray,
    corners: np.ndarray,
) -> np.ndarray:
    """
    Perspective-corrects one cell and removes its steel border.
    """
    canvas_size = int(
        round(
            CELL_SIZE
            / (1.0 - 2.0 * INNER_MARGIN_FRACTION)
        )
    )

    destination = np.float32(
        [
            [0, 0],
            [canvas_size - 1, 0],
            [canvas_size - 1, canvas_size - 1],
            [0, canvas_size - 1],
        ]
    )

    transformation = cv2.getPerspectiveTransform(
        corners.astype(np.float32),
        destination,
    )

    warped = cv2.warpPerspective(
        image,
        transformation,
        (canvas_size, canvas_size),
        flags=cv2.INTER_LINEAR,
    )

    margin = (canvas_size - CELL_SIZE) // 2

    cropped = warped[
        margin:margin + CELL_SIZE,
        margin:margin + CELL_SIZE,
    ]

    return cropped

def warp_big_square(
    image: np.ndarray,
    grid_points: np.ndarray,
    size: int = 1400,
) -> np.ndarray:
    """
    Extract and perspective-correct the entire large quadrat.

    Uses only the four outer corners of the detected steel grid.
    """

    corners = np.float32([
        grid_points[0, 0],  # top-left
        grid_points[0, 2],  # top-right
        grid_points[2, 2],  # bottom-right
        grid_points[2, 0],  # bottom-left
    ])

    destination = np.float32([
        [0, 0],
        [size - 1, 0],
        [size - 1, size - 1],
        [0, size - 1],
    ])

    matrix = cv2.getPerspectiveTransform(
        corners,
        destination,
    )

    big_square = cv2.warpPerspective(
        image,
        matrix,
        (size, size),
        flags=cv2.INTER_LINEAR,
    )

    return big_square

In [39]:
from transformers import Sam3Model, Sam3Processor


device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Sam3Processor.from_pretrained("facebook/sam3")

segmentation_model = Sam3Model.from_pretrained(
    "facebook/sam3",
).to(device)

segmentation_model.eval()

Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 10322.31it/s]


Sam3Model(
  (vision_encoder): Sam3VisionModel(
    (backbone): Sam3ViTModel(
      (embeddings): Sam3ViTEmbeddings(
        (patch_embeddings): Sam3ViTPatchEmbeddings(
          (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (layer_norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (layers): ModuleList(
        (0-31): 32 x Sam3ViTLayer(
          (layer_norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (rotary_emb): Sam3ViTRotaryEmbedding()
          (attention): Sam3ViTRoPEAttention(
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (o_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1024,

In [40]:
def segment_plants(
    rgb_cell,
    prompts=("green leaf",),
    score_threshold=0.25,
    mask_threshold=0.5,
):
    array = np.asarray(rgb_cell)

    # Ensure PIL-compatible uint8 data.
    if array.dtype != np.uint8:
        if array.max() <= 1:
            array = array * 255

        array = np.clip(array, 0, 255).astype(np.uint8)

    image = Image.fromarray(array, mode="RGB")

    combined_mask = np.zeros(
        (image.height, image.width),
        dtype=bool,
    )

    for prompt in prompts:
        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            outputs = segmentation_model(**inputs)

        result = processor.post_process_instance_segmentation(
            outputs,
            threshold=score_threshold,
            mask_threshold=mask_threshold,
            target_sizes=inputs["original_sizes"].tolist(),
        )[0]

        masks = result["masks"]


        if len(masks) > 0:
            prompt_mask = (
                masks.any(dim=0)
                .cpu()
                .numpy()
                .astype(bool)
            )

            combined_mask |= prompt_mask

    # Keep original plant pixels on a white background.
    plant_only = np.full_like(array, 255)
    plant_only[combined_mask] = array[combined_mask]

    return combined_mask, plant_only

In [41]:
all_embeddings = []
valid_paths = []

In [42]:
dino_model.eval()

for start in range(0, len(image_paths), 64):
    batch_paths = image_paths[start:start + 64]

    images = []
    batch_valid_paths = []

    for path in batch_paths:
        image = cv2.imread(str(path))

        if image is None:
            raise ValueError("OpenCV could not read this image")

        grid_points, _, _ = detect_grid(image)

        big_square = warp_big_square(image, grid_points)
        plant_mask, plant_only = segment_plants(big_square)
        # OpenCV loads BGR; DINO expects RGB
        image_rgb = cv2.cvtColor(np.asarray(plant_only), cv2.COLOR_BGR2RGB)

        images.append(image_rgb)
        batch_valid_paths.append(path)

    inputs = processor(images=images, return_tensors="pt").to(dino_model.device)

    with torch.inference_mode():
        outputs = dino_model(**inputs)
        batch_embeddings = F.normalize(
            outputs.pooler_output,
            p=2,
            dim=1
        )

    all_embeddings.append(batch_embeddings.cpu().float().numpy())
    valid_paths.extend(batch_valid_paths)

if not all_embeddings:
    raise RuntimeError("No images were successfully processed.")

embeddings = np.concatenate(all_embeddings, axis=0)

In [43]:
import pandas as pd
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean"
)

labels = clusterer.fit_predict(embeddings)

assignments = pd.DataFrame({
    "image_path": [str(p) for p in valid_paths],
    "cluster_id": labels
}).sort_values(["cluster_id", "image_path"])

assignments.to_csv("cluster_assignments.csv", index=False)

In [46]:
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

X = np.asarray(embeddings, dtype=np.float32)

print("Images:", len(valid_paths))
print("Embedding shape:", X.shape)
print("Finite:", np.isfinite(X).all())
print("Mean dimension std:", X.std(axis=0).mean())

assert len(valid_paths) == len(X)
assert np.isfinite(X).all()

X = normalize(X)

similarities = cosine_similarity(X)
np.fill_diagonal(similarities, -np.inf)

nearest_similarity = similarities.max(axis=1)

print(
    "Nearest-neighbour cosine similarities:",
    np.quantile(nearest_similarity, [0, 0.25, 0.5, 0.75, 1])
)

Images: 470
Embedding shape: (470, 384)
Finite: True
Mean dimension std: 0.018995756
Nearest-neighbour cosine similarities: [0.80088896 0.93482134 0.94506487 0.95228152 0.96794379]


In [47]:
from sklearn.decomposition import PCA
import hdbscan

number_of_components = min(
    50,
    X.shape[0] - 1,
    X.shape[1],
)

X_reduced = PCA(
    n_components=number_of_components,
    random_state=42,
).fit_transform(X)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,  # smallest group you expect
    min_samples=1,       # begin permissively
    metric="euclidean",
    cluster_selection_method="eom",
)

labels = clusterer.fit_predict(X_reduced)

unique_labels, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique_labels, counts)))
print("Noise fraction:", np.mean(labels == -1))

{np.int64(-1): np.int64(64), np.int64(0): np.int64(4), np.int64(1): np.int64(402)}
Noise fraction: 0.13617021276595745
